### Cell 1 -  Setup

In [0]:
# ============================================================
#  02_SILVER_CLEANING — TOURGO DATA PIPELINE
#  Mục đích: Bronze Managed Tables → Cleaning → Silver Managed Tables
# ============================================================

from pyspark.sql.functions import (
    col, to_date, to_timestamp, when, trim, abs as _abs
)
from pyspark.sql.types import DoubleType, IntegerType, BooleanType

# Dùng chung database tourgo đã tạo ở notebook 01
spark.sql("USE tourgo")
print("   Sử dụng Database: tourgo")
print("   Bronze tables: bronze_users, bronze_tours, ...")
print("   Silver tables sẽ tạo: silver_users, silver_tours, ...")

   Sử dụng Database: tourgo
   Bronze tables: bronze_users, bronze_tours, ...
   Silver tables sẽ tạo: silver_users, silver_tours, ...


### Cell 2 - Silver Users

In [0]:
# MỚI (Đã đồng bộ chữ hoa/chữ thường):
from pyspark.sql.functions import upper
# ── USERS ──────────────────────────────────────────────────
print("Cleaning USERS...")


df_u = spark.read.table("bronze_users")
bronze_count = df_u.count()

# df_users_s = df_u \
#     .dropDuplicates(["id"]) \
#     .filter(col("role").isin(["ADMIN", "PROVIDER", "CUSTOMER"])) \
#     .withColumn("is_active",   col("is_active").cast(BooleanType())) \
#     .withColumn("date_joined", to_timestamp(col("date_joined")))


df_users_s = df_u \
    .dropDuplicates(["id"]) \
    .filter(upper(col("role")).isin(["ADMIN", "PROVIDER", "CUSTOMER"])) \
    .withColumn("is_active", col("is_active").cast(BooleanType())) \
    .withColumn("date_joined", to_timestamp(col("date_joined")))

df_users_s.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_users")

silver_count = df_users_s.count()
print(f"   Bronze: {bronze_count:,} -> Silver: {silver_count:,} (dropped: {bronze_count - silver_count})")
df_users_s.show(3, truncate=False)

Cleaning USERS...
   Bronze: 227 -> Silver: 227 (dropped: 0)
+---+--------+---------+-----------------------+
|id |role    |is_active|date_joined            |
+---+--------+---------+-----------------------+
|1  |CUSTOMER|true     |2026-05-20 12:53:18.887|
|2  |CUSTOMER|false    |2026-05-21 17:21:52.753|
|3  |CUSTOMER|false    |2026-05-21 17:22:27.259|
+---+--------+---------+-----------------------+
only showing top 3 rows


### Cell 3 - Silver Tours

In [0]:
# ── TOURS ───────────────────────────────────────────────────
print("Cleaning TOURS...")

df_t = spark.read.table("bronze_tours")
bronze_count = df_t.count()

df_tours_s = df_t \
    .dropDuplicates(["id"]) \
    .withColumn("price", col("price").cast(DoubleType())) \
    .withColumn("slots", col("slots").cast(IntegerType())) \
    .filter(col("price") > 0) \
    .filter(col("slots") >= 0) \
    .withColumn("departure_date", to_date(col("departure_date"))) \
    .withColumn("created_at",     to_timestamp(col("created_at"))) \
    .withColumn("category_names", trim(col("category_names")))

df_tours_s.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_tours")

silver_count = df_tours_s.count()
print(f"   Bronze: {bronze_count:,} -> Silver: {silver_count:,} (dropped: {bronze_count - silver_count})")
df_tours_s.show(3, truncate=False)

Cleaning TOURS...
   Bronze: 779 -> Silver: 779 (dropped: 0)
+---+----------+----------------+-----+--------------+-----+--------+-------------------+--------------+
|id |creator_id|title           |price|departure_date|slots|status  |created_at         |category_names|
+---+----------+----------------+-----+--------------+-----+--------+-------------------+--------------+
|1  |1         |Đà Nẵng         |4.0E7|2026-05-30    |28   |approved|2026-05-18 16:44:50|NULL          |
|2  |2         |Hà Nội          |2.0E7|2026-05-20    |23   |approved|2026-05-18 17:05:34|NULL          |
|3  |3         |Chuyến đi Hội An|5.5E7|2026-06-01    |38   |approved|2026-05-20 09:40:38|NULL          |
+---+----------+----------------+-----+--------------+-----+--------+-------------------+--------------+
only showing top 3 rows


### Cell 4 - Silver Bookings

In [0]:
# ── BOOKINGS ────────────────────────────────────────────────
print("Cleaning BOOKINGS...")

df_b = spark.read.table("bronze_bookings")
bronze_count = df_b.count()

df_bookings_s = df_b \
    .dropDuplicates(["id"]) \
    .filter(col("status").isin(["pending", "confirmed", "cancelled"])) \
    .withColumn("total_price",      col("total_price").cast(DoubleType())) \
    .withColumn("number_of_people", col("number_of_people").cast(IntegerType())) \
    .withColumn("booking_date",     to_date(col("booking_date"))) \
    .withColumn("created_at",       to_timestamp(col("created_at")))

df_bookings_s.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_bookings")

silver_count = df_bookings_s.count()
print(f"   Bronze: {bronze_count:,} -> Silver: {silver_count:,} (dropped: {bronze_count - silver_count})")
df_bookings_s.show(5, truncate=False)

Cleaning BOOKINGS...
   Bronze: 1,326 -> Silver: 1,326 (dropped: 0)
+---+-------+-------+----------------+-----------+------------+---------+-------------------+
|id |user_id|tour_id|number_of_people|total_price|booking_date|status   |created_at         |
+---+-------+-------+----------------+-----------+------------+---------+-------------------+
|1  |2      |1      |1               |4.0E7      |2026-05-30  |confirmed|2026-05-18 16:45:27|
|2  |2      |1      |3               |4.0E7      |2026-05-30  |confirmed|2026-05-18 16:59:53|
|3  |2      |2      |1               |2.0E7      |2026-05-20  |confirmed|2026-05-18 17:08:31|
|4  |2      |2      |1               |2.0E7      |2026-05-20  |pending  |2026-05-18 17:08:47|
|5  |11     |3      |1               |5.5E7      |2026-06-01  |confirmed|2026-05-23 15:32:56|
+---+-------+-------+----------------+-----------+------------+---------+-------------------+
only showing top 5 rows


### Cell 5 - Silver Payments

In [0]:
# ── PAYMENTS ────────────────────────────────────────────────
print("Cleaning PAYMENTS...")

df_p = spark.read.table("bronze_payments")
bronze_count = df_p.count()

df_payments_s = df_p \
    .dropDuplicates(["id"]) \
    .filter(col("payment_method").isin(["VietQR", "VNPAY", "CASH", "BANK_TRANSFER"])) \
    .withColumn("amount",     col("amount").cast(DoubleType())) \
    .withColumn("created_at", to_timestamp(col("created_at")))

df_payments_s.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_payments")

silver_count = df_payments_s.count()
print(f"   Bronze: {bronze_count:,} -> Silver: {silver_count:,} (dropped: {bronze_count - silver_count})")
df_payments_s.show(5, truncate=False)

Cleaning PAYMENTS...
   Bronze: 794 -> Silver: 794 (dropped: 0)
+---+----------+---------+--------------+-------+-------------------+
|id |booking_id|amount   |payment_method|status |created_at         |
+---+----------+---------+--------------+-------+-------------------+
|151|263       |2000000.0|CASH          |SUCCESS|2026-08-24 11:44:52|
|171|291       |6000000.0|BANK_TRANSFER |SUCCESS|2025-06-14 06:29:34|
|200|334       |1.4E7    |VNPAY         |SUCCESS|2026-11-19 21:52:43|
|102|185       |2500000.0|BANK_TRANSFER |SUCCESS|2025-01-20 20:46:23|
|129|228       |1000000.0|CASH          |SUCCESS|2025-05-16 06:55:09|
+---+----------+---------+--------------+-------+-------------------+
only showing top 5 rows


### Cell 6 - Silver Revenues

In [0]:
# ── REVENUES ────────────────────────────────────────────────
# Dùng tolerance 0.01 thay vì == để tránh floating point error
print("Cleaning REVENUES...")

df_r = spark.read.table("bronze_revenues")
bronze_count = df_r.count()

df_revenues_s = df_r \
    .dropDuplicates(["id"]) \
    .withColumn("total_amount",  col("total_amount").cast(DoubleType())) \
    .withColumn("creator_share", col("creator_share").cast(DoubleType())) \
    .withColumn("admin_share",   col("admin_share").cast(DoubleType())) \
    .withColumn("created_at",    to_timestamp(col("created_at"))) \
    .filter(
        _abs(col("creator_share") + col("admin_share") - col("total_amount")) < 0.01
    )

df_revenues_s.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_revenues")

silver_count = df_revenues_s.count()
print(f"   Bronze: {bronze_count:,} -> Silver: {silver_count:,} (dropped: {bronze_count - silver_count})")
df_revenues_s.show(5, truncate=False)

Cleaning REVENUES...
   Bronze: 788 -> Silver: 788 (dropped: 0)
+---+----------+----------+------------+-------------+-----------+-------------------+
|id |payment_id|creator_id|total_amount|creator_share|admin_share|created_at         |
+---+----------+----------+------------+-------------+-----------+-------------------+
|28 |36        |185       |750000.0    |675000.0     |75000.0    |2026-05-31 08:45:20|
|29 |37        |193       |2000000.0   |1800000.0    |200000.0   |2026-05-31 08:45:20|
|30 |38        |164       |450000.0    |405000.0     |45000.0    |2026-05-31 08:45:20|
|33 |41        |215       |5.0E7       |4.5E7        |5000000.0  |2026-05-31 08:45:20|
|93 |101       |215       |5.0E7       |4.5E7        |5000000.0  |2026-05-31 08:45:21|
+---+----------+----------+------------+-------------+-----------+-------------------+
only showing top 5 rows


### Cell 7 - Silver Reviews

In [0]:
# ── REVIEWS ─────────────────────────────────────────────────
print("Cleaning REVIEWS...")

df_rv = spark.read.table("bronze_reviews")
bronze_count = df_rv.count()

df_reviews_s = df_rv \
    .dropDuplicates(["id"]) \
    .withColumn("rating", col("rating").cast(IntegerType())) \
    .filter(col("rating").isNotNull()) \
    .filter((col("rating") >= 1) & (col("rating") <= 5)) \
    .withColumn("created_at", to_timestamp(col("created_at")))

df_reviews_s.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_reviews")

silver_count = df_reviews_s.count()
print(f"   Bronze: {bronze_count:,} -> Silver: {silver_count:,} (dropped: {bronze_count - silver_count})")
df_reviews_s.show(3, truncate=False)

Cleaning REVIEWS...
   Bronze: 272 -> Silver: 272 (dropped: 0)
+---+-------+-------+------+-------------------+
|id |user_id|tour_id|rating|created_at         |
+---+-------+-------+------+-------------------+
|28 |28     |253    |4     |2026-10-23 11:40:10|
|93 |75     |284    |5     |2026-09-05 21:22:37|
|33 |124    |196    |5     |2026-06-29 07:42:04|
+---+-------+-------+------+-------------------+
only showing top 3 rows


### Cell 8 - Comparison Report (chụp screenshot)

In [0]:
# ============================================================
#  BRONZE vs SILVER COMPARISON — chụp screenshot cell này
# ============================================================

TABLES = ['users', 'tours', 'bookings', 'payments', 'revenues', 'reviews']

print("\n" + "="*55)
print("BRONZE vs SILVER COMPARISON")
print("="*55)
print(f"{'Table':<12} | {'Bronze':>8} | {'Silver':>8} | {'Dropped':>8}")
print("-"*45)

for t in TABLES:
    try:
        b = spark.read.table(f"bronze_{t}").count()
        s = spark.read.table(f"silver_{t}").count()
        flag = "--(ERROR)--" if (b - s) > 0 else "-OK-"
        print(f"{t:<12} | {b:>8,} | {s:>8,} | {b-s:>8,}{flag}")
    except Exception as e:
        print(f"{t:<12} | {'ERR':>8} | {'ERR':>8} | {str(e)[:25]}")

print("="*55)
print("Silver Layer hoàn tất — sẵn sàng cho Day 3 (Gold)")


BRONZE vs SILVER COMPARISON
Table        |   Bronze |   Silver |  Dropped
---------------------------------------------
users        |      227 |      227 |        0-OK-
tours        |      779 |      779 |        0-OK-
bookings     |    1,326 |    1,326 |        0-OK-
payments     |      794 |      794 |        0-OK-
revenues     |      788 |      788 |        0-OK-
reviews      |      272 |      272 |        0-OK-
Silver Layer hoàn tất — sẵn sàng cho Day 3 (Gold)


### Cell 10 - Gửi cho Khánh

In [0]:
# # Chạy cell này và copy kết quả gửi cho [D] Khang
# df_users  = spark.read.table("silver_users")
# df_tours  = spark.read.table("silver_tours")

# print("SAMPLE USER IDs (gửi cho Khánh):")
# df_users.select("id").show(10, truncate=False)

# print("SAMPLE TOUR IDs + PRICES (gửi cho Khánh):")
# df_tours.select("id", "price").show(10, truncate=False)

In [0]:
# ============================================================
#  COPY KẾT QUẢ NÀY GỬI CHO [D] KHÁNH
#  Để Khang điền vào streaming_simulator.py


print("=" * 50)
print("DATA CHO [D] KHÁNH — streaming_simulator.py")
print("=" * 50)

df_users_info = spark.read.table("silver_users") \
    .filter(col("role") == "CUSTOMER") \
    .select("id") \
    .limit(10)

df_tours_info = spark.read.table("silver_tours") \
    .select("id", "price") \
    .limit(10)

user_ids = [row["id"] for row in df_users_info.collect()]
tour_rows = df_tours_info.collect()
tour_ids  = [row["id"]    for row in tour_rows]
prices    = list(set([int(row["price"]) for row in tour_rows]))

print(f"SAMPLE_USER_IDS = {user_ids}")
print(f"SAMPLE_TOUR_IDS = {tour_ids}")
print(f"SAMPLE_PRICES   = {prices}")
print("\n-> Copy 3 dòng trên paste vào streaming_simulator.py thay cho placeholder")

DATA CHO [D] KHÁNH — streaming_simulator.py
SAMPLE_USER_IDS = [3, 19, 2, 7, 5, 1]
SAMPLE_TOUR_IDS = [33, 151, 28, 29, 30, 171, 93, 564, 364, 262]
SAMPLE_PRICES   = [3000000, 1000000, 1500000, 500000, 250000, 350000, 750000, 1231123]

-> Copy 3 dòng trên paste vào streaming_simulator.py thay cho placeholder


In [0]:
print(5)

5
